<div>
<img src="img/RDM_course_2024_alpha_crop.png" height="150"/>
</div>

# add-raman-spectroscopy

## development notebook

---

In [1]:
import rich
from rich.pretty import Pretty
import uuid
from datetime import datetime

import matplotlib
matplotlib.use("qtagg")  # Preferred backend, alternative is TkAgg
# %matplotlib ipympl     # Not working properly with matplotlib buttons
from Ramacropy.Ramacropy import RamanSpectra

from modules.fairvibspec import FAIRVibSpec, Experiment, Measurement, VibSpecType, MeasurementTypes, Detection, Dataset, Series, UnitDefinition, SIFParameters, Analysis

In [2]:
def pretty_print(output):
    rich.print(Pretty(output, max_length=10))

### Initialisation of the RamanSpectra object with typical SIF file

Ramacropy initialises from a .csv or .sif file. The .sif file is the output from the Raman spectrometer. As the laser wavelength is not available from the .sif file, we need to provide it manually.

In [3]:
my_spec = RamanSpectra("./data/Simon/298K-ADY79-10s-10acc.sif", laser_wavelength=785.0)

The library uses open-source package `sif_parser` to read the .sif file and creates a few useful attributes like numpy arrays for the x and y data, including a copy of the raw data, as well as the metadata as an ordered dictionary.

In [4]:
print(f"Raman shift:\n{my_spec.RamanShift}")      # the x axis of your data, contains the shift at each y point
print(f"Timestamp:\n{my_spec.TimeStamp}")         # the time data of each spectrum in the series
print(f"Spectral data:\n{my_spec.SpectralData}")  # the actual y data of your file
print(f"Raw data:\n{my_spec.RawData}")            # the same as spectral data, but doesn't change if you perform processing (only .pkl and .sif)
print(f"Metadata:\n{my_spec.SpectralInfo}")       # this is the metadata from andor see sif_parser for more info (only .pkl and .sif)
print(f"Path to file:\n{my_spec.directory}")      # original path to file
print(f"File name:\n{my_spec.filelab}")           # original file name

Raman shift:
[-11520.37537197 -11519.06937488 -11517.76365559 ... -10315.98681346
 -10314.91979467 -10313.85297109]
Timestamp:
[0.]
Spectral data:
[[365.9826 ]
 [361.21832]
 [355.99783]
 ...
 [370.46017]
 [369.71353]
 [370.78873]]
Raw data:
[[365.9826 ]
 [361.21832]
 [355.99783]
 ...
 [370.46017]
 [369.71353]
 [370.78873]]
Metadata:
OrderedDict({'SifVersion': 65567, 'ExperimentTime': 1709825147, 'DetectorTemperature': -50.0, 'ExposureTime': 10.0, 'CycleTime': 10.0, 'AccumulatedCycleTime': 10.0, 'AccumulatedCycles': 10, 'StackCycleTime': 99.9999, 'PixelReadoutTime': 1e-05, 'GainDAC': 0.0, 'GateWidth': 0.0, 'GratingBlaze': 3.225e-05, 'DetectorType': 'DU420_OE', 'DetectorDimensions': (1024, 255), 'OriginalFilename': b'C:\\Users\\lo_raman\\Desktop\\100azo temp\\298K-ADY79-10s-10acc.sif', 'ShutterTime': (0.0, 0.0), 'spectrograph': '999', 'GateGain': 0.0, 'GateDelay': 0.0, 'SifCalbVersion': 65540, 'Calibration_data': [412.192061638435, 0.0221962737363239, -1.16645505452665e-06, 8.43925774014

### Cropping the data to the region of interest

Weirdly, the y-axis is longer than the x-axis, which Ramacropy does not accept. For now, we just crop the y-axis to the same length as the x-axis in the region of interest. Have to confirm with Simon, though.

In [5]:
my_spec.SpectralData = my_spec.SpectralData[601:1625]
my_spec.RawData = my_spec.RawData[601:1625]

### Creating the now Raman-capable data model

__Of course, this will be done automatically and in the background in the future__

I allowed myself to rename the old `IRAnalysis` repository and root object of the data model to `FAIRVibSpec` to better reflect the scope of the package.

In [6]:
data_model = FAIRVibSpec(
    datetime_created=str(datetime.now()),
    contributors=[
        "Simon Krause <s.krause@fkf.mpg.de>",
        "Torsten Giess <torsten.giess@ibtb.uni-stuttgart.de>"
    ]
)
pretty_print(data_model)

FAIRVibSpec(
    datetime_created='2025-02-06 09:29:27.010805',
    datetime_modified=None,
    contributors=['Simon Krause <s.krause@fkf.mpg.de>', 'Torsten Giess <torsten.giess@ibtb.uni-stuttgart.de>'],
    experiment=None,
    ld_id='md:FAIRVibSpec/4b16a418-cac8-427d-9d77-685581edfedf',
    ld_type=['md:FAIRVibSpec'],
    ld_context={'md': 'http://mdmodel.net/'}
)

In [7]:
my_experiment = Experiment(
    name=my_spec.filelab,
)
pretty_print(my_experiment)

Experiment(
    name='298K-ADY79-10s-10acc',
    varied_parameter=None,
    static_parameters=None,
    measurements=[],
    analysis=[],
    results=None,
    ld_id='md:Experiment/99b45349-12d1-4ab5-a2a7-8d46dc363065',
    ld_type=['md:Experiment'],
    ld_context={'md': 'http://mdmodel.net/'}
)

In [8]:
new_measurement = Measurement(
    id=str(uuid.uuid4()),
    vib_spec_type=VibSpecType.RAMAN.value,
    measurement_type=MeasurementTypes.SAMPLE.value,
    detection=Detection.INTENSITY.value,
)
pretty_print(new_measurement)


Measurement(
    id='e469b4eb-2028-4ac8-bd59-62751f029ec5',
    vib_spec_type=<VibSpecType.RAMAN: 'raman'>,
    detection=<Detection.INTENSITY: 'intensity'>,
    name=None,
    varied_parameter_value=None,
    measurement_type=<MeasurementTypes.SAMPLE: 'sample'>,
    measurement_data=None,
    static_parameters=None,
    instrument_parameters=None,
    ld_id='md:Measurement/4b1ce0f2-4674-4c9f-86e8-11bd6738b680',
    ld_type=['md:Measurement'],
    ld_context={'md': 'http://mdmodel.net/', 'id': {'@type': '@id'}}
)

In [9]:
new_dataset = Dataset(
    timestamp=str(datetime.fromtimestamp(my_spec.SpectralInfo["ExperimentTime"])),
    x_axis=Series(
        data_array=my_spec.RamanShift,
        unit=UnitDefinition(
            name=my_spec.SpectralInfo["FrameAxis"].decode("utf-8"),
        )
    ),
    y_axis=Series(
        data_array=my_spec.SpectralData,
        unit=UnitDefinition(
            name=my_spec.SpectralInfo["DataType"].decode("utf-8"),
        )
    )
)
pretty_print(new_dataset)

Dataset(
    timestamp='2024-03-07 16:25:47',
    x_axis=Series(
        data_array=[
            -11520.375371974445,
            -11519.069374875451,
            -11517.763655590743,
            -11516.458214023514,
            -11515.153050077008,
            -11513.848163654486,
            -11512.543554659256,
            -11511.239222994662,
            -11509.935168564076,
            -11508.631391270914,
            ... +1014
        ],
        unit=UnitDefinition(
            id=None,
            name='Raman shift',
            base_units=[],
            ld_id='md:UnitDefinition/d99ea04d-1424-4f36-aa31-4e2aeb236495',
            ld_type=['md:UnitDefinition'],
            ld_context={'md': 'http://mdmodel.net/'}
        ),
        ld_id='md:Series/429ff517-5cbe-4ea5-b97d-07b31c48e731',
        ld_type=['md:Series'],
        ld_context={'md': 'http://mdmodel.net/'}
    ),
    y_axis=Series(
        data_array=[
            738.9502563476562,
            738.7369995117188,
            738.7333374023438,
            738.2753295898438,
            734.1027221679688,
            740.3294677734375,
            740.7662353515625,
            746.310302734375,
            754.7574462890625,
            751.2701416015625,
            ... +1014
        ],
        unit=UnitDefinition(
            id=None,
            name='Counts',
            base_units=[],
            ld_id='md:UnitDefinition/059295f8-92e3-43b8-bac0-152d2b5ee179',
            ld_type=['md:UnitDefinition'],
            ld_context={'md': 'http://mdmodel.net/'}
        ),
        ld_id='md:Series/b22a3573-86e4-4a0d-b5a9-baf526d1995c',
        ld_type=['md:Series'],
        ld_context={'md': 'http://mdmodel.net/'}
    ),
    ld_id='md:Dataset/0acfdc33-6ac9-436f-b04e-bc6aa76697d3',
    ld_type=['md:Dataset'],
    ld_context={'md': 'http://mdmodel.net/'}
)

Up to this point, a lot is the same as before. However, we now have the option to add instrument parameters to the measurement object. Currently, this is only implemented for SIF files, but will be extended to include other file types (as needed) in the future.

In [10]:
raman_metadata = SIFParameters(
    sif_version=my_spec.SpectralInfo["SifVersion"],
    experiment_time=my_spec.SpectralInfo["ExperimentTime"],
    detector_temperature=my_spec.SpectralInfo["DetectorTemperature"],
    exposure_time=my_spec.SpectralInfo["ExposureTime"],
    cycle_time=my_spec.SpectralInfo["CycleTime"],
    accumulated_cycle_time=my_spec.SpectralInfo["AccumulatedCycleTime"],
    accumulated_cycles=my_spec.SpectralInfo["AccumulatedCycles"],
    stack_cycle_time=my_spec.SpectralInfo["StackCycleTime"],
    pixel_readout_time=my_spec.SpectralInfo["PixelReadoutTime"],
    gain_dac=my_spec.SpectralInfo["GainDAC"],
    gate_width=my_spec.SpectralInfo["GateWidth"],
    grating_blaze=my_spec.SpectralInfo["GratingBlaze"],
    detector_type=my_spec.SpectralInfo["DetectorType"],
    detector_dimensions=my_spec.SpectralInfo["DetectorDimensions"],
    original_filename=my_spec.SpectralInfo["OriginalFilename"].decode("utf-8"),
    shutter_time=my_spec.SpectralInfo["ShutterTime"],
    spectrograph=my_spec.SpectralInfo["spectrograph"],
    gate_gain=my_spec.SpectralInfo["GateGain"],
    gate_delay=my_spec.SpectralInfo["GateDelay"],
    sif_calibration_version=my_spec.SpectralInfo["SifCalbVersion"],
    calibration_data=my_spec.SpectralInfo["Calibration_data"],
    raman_excitation_wavelength=my_spec.SpectralInfo["RamanExWavelength"],
    frame_axis=my_spec.SpectralInfo["FrameAxis"].decode("utf-8"),
    data_type=my_spec.SpectralInfo["DataType"].decode("utf-8"),
    image_axis=my_spec.SpectralInfo["ImageAxis"].decode("utf-8"),
    number_of_frames=my_spec.SpectralInfo["NumberOfFrames"],
    number_of_sub_images=my_spec.SpectralInfo["NumberOfSubImages"],
    total_length=my_spec.SpectralInfo["TotalLength"],
    image_length=my_spec.SpectralInfo["ImageLength"],
    xbin=my_spec.SpectralInfo["xbin"],
    ybin=my_spec.SpectralInfo["ybin"],
    timestamp_of_0=my_spec.SpectralInfo["timestamp_of_0"],
    size=my_spec.SpectralInfo["size"],
    tile=str(my_spec.SpectralInfo["tile"]),
    offset=my_spec.SpectralInfo["offset"],
)
pretty_print(raman_metadata)

SIFParameters(
    sif_version=65567,
    experiment_time=1709825147,
    detector_temperature=-50.0,
    exposure_time=10.0,
    cycle_time=10.0,
    accumulated_cycle_time=10.0,
    accumulated_cycles=10,
    stack_cycle_time=99.9999,
    pixel_readout_time=1e-05,
    gain_dac=0.0,
    gate_width=0.0,
    grating_blaze=3.225e-05,
    detector_type='DU420_OE',
    detector_dimensions=[1024, 255],
    original_filename='C:\\Users\\lo_raman\\Desktop\\100azo temp\\298K-ADY79-10s-10acc.sif',
    shutter_time=[0.0, 0.0],
    spectrograph='999',
    gate_gain=0.0,
    gate_delay=0.0,
    sif_calibration_version=65540,
    calibration_data=[412.192061638435, 0.0221962737363239, -1.16645505452665e-06, 8.43925774014707e-11],
    raman_excitation_wavelength=433.0,
    frame_axis='Raman shift',
    data_type='Counts',
    image_axis='Pixel number',
    number_of_frames=1,
    number_of_sub_images=1,
    total_length=3482,
    image_length=3482,
    xbin=1,
    ybin=255,
    timestamp_of_0=0,
    size=[3482, 1],
    tile="[('raw', (0, 0, 3482, 1), 2881, ('F;32F', 0, 1))]",
    offset=2881,
    ld_id='md:SIFParameters/824b49fb-a37f-4d0d-bc87-cc02aef0270b',
    ld_type=['md:SIFParameters'],
    ld_context={'md': 'http://mdmodel.net/'}
)

In [11]:
new_measurement.measurement_data = new_dataset
new_measurement.instrument_parameters = raman_metadata

my_experiment.measurements.append(new_measurement)

data_model.experiment = my_experiment

pretty_print(data_model)

FAIRVibSpec(
    datetime_created='2025-02-06 09:29:27.010805',
    datetime_modified=None,
    contributors=['Simon Krause <s.krause@fkf.mpg.de>', 'Torsten Giess <torsten.giess@ibtb.uni-stuttgart.de>'],
    experiment=Experiment(
        name='298K-ADY79-10s-10acc',
        varied_parameter=None,
        static_parameters=None,
        measurements=[
            Measurement(
                id='e469b4eb-2028-4ac8-bd59-62751f029ec5',
                vib_spec_type=<VibSpecType.RAMAN: 'raman'>,
                detection=<Detection.INTENSITY: 'intensity'>,
                name=None,
                varied_parameter_value=None,
                measurement_type=<MeasurementTypes.SAMPLE: 'sample'>,
                measurement_data=Dataset(
                    timestamp='2024-03-07 16:25:47',
                    x_axis=Series(
                        data_array=[
                            -11520.375371974445,
                            -11519.069374875451,
                            -11517.763655590743,
                            -11516.458214023514,
                            -11515.153050077008,
                            -11513.848163654486,
                            -11512.543554659256,
                            -11511.239222994662,
                            -11509.935168564076,
                            -11508.631391270914,
                            ... +1014
                        ],
                        unit=UnitDefinition(
                            id=None,
                            name='Raman shift',
                            base_units=[],
                            ld_id='md:UnitDefinition/d99ea04d-1424-4f36-aa31-4e2aeb236495',
                            ld_type=['md:UnitDefinition'],
                            ld_context={'md': 'http://mdmodel.net/'}
                        ),
                        ld_id='md:Series/429ff517-5cbe-4ea5-b97d-07b31c48e731',
                        ld_type=['md:Series'],
                        ld_context={'md': 'http://mdmodel.net/'}
                    ),
                    y_axis=Series(
                        data_array=[
                            738.9502563476562,
                            738.7369995117188,
                            738.7333374023438,
                            738.2753295898438,
                            734.1027221679688,
                            740.3294677734375,
                            740.7662353515625,
                            746.310302734375,
                            754.7574462890625,
                            751.2701416015625,
                            ... +1014
                        ],
                        unit=UnitDefinition(
                            id=None,
                            name='Counts',
                            base_units=[],
                            ld_id='md:UnitDefinition/059295f8-92e3-43b8-bac0-152d2b5ee179',
                            ld_type=['md:UnitDefinition'],
                            ld_context={'md': 'http://mdmodel.net/'}
                        ),
                        ld_id='md:Series/b22a3573-86e4-4a0d-b5a9-baf526d1995c',
                        ld_type=['md:Series'],
                        ld_context={'md': 'http://mdmodel.net/'}
                    ),
                    ld_id='md:Dataset/0acfdc33-6ac9-436f-b04e-bc6aa76697d3',
                    ld_type=['md:Dataset'],
                    ld_context={'md': 'http://mdmodel.net/'}
                ),
                static_parameters=None,
                instrument_parameters=SIFParameters(
                    sif_version=65567,
                    experiment_time=1709825147,
                    detector_temperature=-50.0,
                    exposure_time=10.0,
                    cycle_time=10.0,
                    accumulated_cycle_time=10.0,
                    accumulated_cycles=10,
                    stack_cycle_time=99.9999,
                    pixel_readout_time=1e-05,
               

### Let's see (part of) what Ramacropy can do

Sadly, the widget-style `ipympl` backend of matplotlib does not work with the interactive plots without complete refactoring specifically for Jupter Notebooks. So we have to rely on pop-up windows for the interactive plots based on either `QtAgg` or `TkAgg`.

In [19]:
my_spec.plot_few()

In [13]:
my_spec.baseline(interactive=True, debounce_interval=10)

### Adding the analysis/processing to the data model and saving it as JSON

__Support for the Raman Allotrope Simple Model will be added in the future__

In [14]:
analysis = Analysis(
    sample_reference=new_measurement.id,
    corrected_data=Dataset(
        timestamp=str(datetime.now()),
        x_axis=Series(
            data_array=my_spec.RamanShift,
            unit=UnitDefinition(name=my_spec.SpectralInfo["FrameAxis"].decode("utf-8")),
        ),
        y_axis=Series(
            data_array=my_spec.SpectralData,
            unit=UnitDefinition(name=my_spec.SpectralInfo["DataType"].decode("utf-8")),
        ),
    )
)
pretty_print(analysis)

Analysis(
    sample_reference='e469b4eb-2028-4ac8-bd59-62751f029ec5',
    background_reference=None,
    corrected_data=Dataset(
        timestamp='2025-02-06 09:40:17.167493',
        x_axis=Series(
            data_array=[
                -11520.375371974445,
                -11519.069374875451,
                -11517.763655590743,
                -11516.458214023514,
                -11515.153050077008,
                -11513.848163654486,
                -11512.543554659256,
                -11511.239222994662,
                -11509.935168564076,
                -11508.631391270914,
                ... +1014
            ],
            unit=UnitDefinition(
                id=None,
                name='Raman shift',
                base_units=[],
                ld_id='md:UnitDefinition/da7be858-ae0a-4fa7-ba30-599bdaa01e7c',
                ld_type=['md:UnitDefinition'],
                ld_context={'md': 'http://mdmodel.net/'}
            ),
            ld_id='md:Series/a51462c8-55b4-4b34-922a-1cd9365d1d3f',
            ld_type=['md:Series'],
            ld_context={'md': 'http://mdmodel.net/'}
        ),
        y_axis=Series(
            data_array=[
                1397.6025390625,
                1396.7669677734375,
                1396.131591796875,
                1395.032470703125,
                1390.209228515625,
                1395.7760009765625,
                1395.543212890625,
                1400.4083251953125,
                1408.1671142578125,
                1403.9820556640625,
                ... +1014
            ],
            unit=UnitDefinition(
                id=None,
                name='Counts',
                base_units=[],
                ld_id='md:UnitDefinition/7595a4ff-bdd5-420d-828c-68b34778b684',
                ld_type=['md:UnitDefinition'],
                ld_context={'md': 'http://mdmodel.net/'}
            ),
            ld_id='md:Series/9badd0b1-8148-4abd-af2e-b2ddae508b10',
            ld_type=['md:Series'],
            ld_context={'md': 'http://mdmodel.net/'}
        ),
        ld_id='md:Dataset/5620b546-3a9c-457e-9d1e-f94133a38039',
        ld_type=['md:Dataset'],
        ld_context={'md': 'http://mdmodel.net/'}
    ),
    baseline=None,
    bands=[],
    measurement_results=[],
    ld_id='md:Analysis/25629855-2568-4af8-b6cd-49d985f4186f',
    ld_type=['md:Analysis'],
    ld_context={'md': 'http://mdmodel.net/'}
)

In [15]:
data_model.experiment.analysis.append(analysis)
pretty_print(data_model)

FAIRVibSpec(
    datetime_created='2025-02-06 09:29:27.010805',
    datetime_modified=None,
    contributors=['Simon Krause <s.krause@fkf.mpg.de>', 'Torsten Giess <torsten.giess@ibtb.uni-stuttgart.de>'],
    experiment=Experiment(
        name='298K-ADY79-10s-10acc',
        varied_parameter=None,
        static_parameters=None,
        measurements=[
            Measurement(
                id='e469b4eb-2028-4ac8-bd59-62751f029ec5',
                vib_spec_type=<VibSpecType.RAMAN: 'raman'>,
                detection=<Detection.INTENSITY: 'intensity'>,
                name=None,
                varied_parameter_value=None,
                measurement_type=<MeasurementTypes.SAMPLE: 'sample'>,
                measurement_data=Dataset(
                    timestamp='2024-03-07 16:25:47',
                    x_axis=Series(
                        data_array=[
                            -11520.375371974445,
                            -11519.069374875451,
                            -11517.763655590743,
                            -11516.458214023514,
                            -11515.153050077008,
                            -11513.848163654486,
                            -11512.543554659256,
                            -11511.239222994662,
                            -11509.935168564076,
                            -11508.631391270914,
                            ... +1014
                        ],
                        unit=UnitDefinition(
                            id=None,
                            name='Raman shift',
                            base_units=[],
                            ld_id='md:UnitDefinition/d99ea04d-1424-4f36-aa31-4e2aeb236495',
                            ld_type=['md:UnitDefinition'],
                            ld_context={'md': 'http://mdmodel.net/'}
                        ),
                        ld_id='md:Series/429ff517-5cbe-4ea5-b97d-07b31c48e731',
                        ld_type=['md:Series'],
                        ld_context={'md': 'http://mdmodel.net/'}
                    ),
                    y_axis=Series(
                        data_array=[
                            738.9502563476562,
                            738.7369995117188,
                            738.7333374023438,
                            738.2753295898438,
                            734.1027221679688,
                            740.3294677734375,
                            740.7662353515625,
                            746.310302734375,
                            754.7574462890625,
                            751.2701416015625,
                            ... +1014
                        ],
                        unit=UnitDefinition(
                            id=None,
                            name='Counts',
                            base_units=[],
                            ld_id='md:UnitDefinition/059295f8-92e3-43b8-bac0-152d2b5ee179',
                            ld_type=['md:UnitDefinition'],
                            ld_context={'md': 'http://mdmodel.net/'}
                        ),
                        ld_id='md:Series/b22a3573-86e4-4a0d-b5a9-baf526d1995c',
                        ld_type=['md:Series'],
                        ld_context={'md': 'http://mdmodel.net/'}
                    ),
                    ld_id='md:Dataset/0acfdc33-6ac9-436f-b04e-bc6aa76697d3',
                    ld_type=['md:Dataset'],
                    ld_context={'md': 'http://mdmodel.net/'}
                ),
                static_parameters=None,
                instrument_parameters=SIFParameters(
                    sif_version=65567,
                    experiment_time=1709825147,
                    detector_temperature=-50.0,
                    exposure_time=10.0,
                    cycle_time=10.0,
                    accumulated_cycle_time=10.0,
                    accumulated_cycles=10,
                    stack_cycle_time=99.9999,
                    pixel_readout_time=1e-05,
               

In [16]:
with open(f"./data/Simon/{my_spec.filelab}.json", "w") as f:
    f.write(
        data_model.model_dump_json(
            indent=2,
            exclude_none=True
        )
    )
    print("Success! 🎉")

Success! 🎉


---

### \~\~\~under heavy development\~\~\~

#### 1) Time-course data from Raman will be added

Currently no data to test this with...

In [17]:
my_spec.plot_kinetic()

ValueError: This is not a kinetic file, it is a single file. Please use the appropriate function to plot it

#### 2) Normalisation and integration will be added
__They are currently broken even in original code__

In [18]:
my_spec.normalise(method="area",interactive = True)

Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniconda/base/envs/iranalysis/lib/python3.13/site-packages/matplotlib/cbook.py", line 361, in process
    func(*args, **kwargs)
    ~~~~^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/envs/iranalysis/lib/python3.13/site-packages/matplotlib/widgets.py", line 592, in <lambda>
    return self._observers.connect('changed', lambda val: func(val))
                                                          ~~~~^^^^^
  File "/Users/torstengiess/Documents/GitHub/Ramacropy/Ramacropy/Utils.py", line 313, in update
    start.set_xdata(start_slider.val)
    ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/envs/iranalysis/lib/python3.13/site-packages/matplotlib/lines.py", line 1290, in set_xdata
    raise RuntimeError('x must be a sequence')
RuntimeError: x must be a sequence
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniconda/base/envs/iranalysis/lib/python3.13/si

In [ ]:
my_spec.integrate(interactive=True)

#### 3) Full integration with `FAIRVibSpec` will be added

- Data model overhaul
- Full Raman integration
- Real-world testing with
  1. Pyridine FTIR of silica materials from Johanna Bruckner
  2. FTIR and Raman of chitosan from Achim Weber
  3. FTIR and Raman of MOFs from Simon Krause
- IR refactoring
- Documentation update
- Package release